# Graph-Based Premise Recommendation in Lean/mathlib
## Import Modules

In [1]:
!pip install torch_geometric sentence-transformers

# Detect current Colab PyTorch version and install libraries
import torch
pt_version = torch.__version__
print(f"PyTorch version: {pt_version}")

!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-{pt_version}.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.1 MB/s eta 0:00:00
PyTorch version: 2.10.0+cu128
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 102.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 110.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 125.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 69.5 MB/s eta 0:00:00


In [2]:
# import time
# import random
import os
import pandas as pd
import networkx as nx
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import SAGEConv # GATv2Conv
from torch_geometric.loader import LinkNeighborLoader
# from torch.utils.data import DataLoader
from scipy.stats import pearsonr
from sentence_transformers import SentenceTransformer
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


## Load Dataset

In [4]:
from google.colab import files, drive
# uploaded = files.upload()

drive.mount('/content/drive')

# Set data path within Drive (Modify to match your actual upload path)
# qe.g.: /content/drive/MyDrive/MathlibGraph_Data/v2/...
base_path = '/content/drive/MyDrive/CS471'

Mounted at /content/drive


In [90]:
# Data file path (Modify according to environment)

# mathlib_node = "mathlib_nodes.csv"
# mathlib_edge = "mathlib_edges.csv"
# node = "nodes.csv"
# edge = "edges.csv"
# declaration_node = "declaration_nodes.csv"
# declaration_metrics = "declaration_metrics.csv"
# namespace_node = "namespace_nodes.csv"
# namespace_edge = "namespace_edges.csv"
# module_node = "module_nodes.csv"
# module_edge = "module_edges.csv"

###############################################################
# IMPORTANT! Choose whether to use all data or explicit data
# Keep the parts you will use and delete the parts you will not use

# Whole data
# node = os.path.join(base_path, "nodes_with_types_full.csv")
# edge = os.path.join(base_path, "mathlib_edges.csv")

# Explicit data only
node = os.path.join(base_path, "nodes_with_types_explicit.csv")
edge = os.path.join(base_path, "mathlib_edges_explicit.csv")
###############################################################


In [9]:
# Load data
df_nodes = pd.read_csv(node)
df_edges = pd.read_csv(edge)

print(f"Graph has {len(df_nodes)} nodes, {len(df_edges)} edges.")

Graph has 284304 nodes, 2178534 edges.


## Create Graphs

In [10]:
G = nx.DiGraph()
G.add_nodes_from(df_nodes['name'])
G.add_edges_from(zip(df_edges['source'], df_edges['target']))

# Filter Edge-free nodes
# valid_nodes = set(G.nodes())
# df_nodes = df_nodes[df_nodes['name'].isin(valid_nodes)].copy()

In [92]:
# Kind distribution of selected graph

kind_summary = (
    df_nodes['kind']
    .value_counts()
    .rename_axis('kind')
    .reset_index(name='count')
)

kind_summary['ratio'] = kind_summary['count'] / kind_summary['count'].sum()

display(
    kind_summary.style
    .format({'ratio': '{:.4f}'})
    .bar(subset=['ratio'])
)

,kind,count,ratio
0,theorem,225833,0.7943
1,definition,46079,0.1621
2,abbrev,6080,0.0214
3,constructor,4446,0.0156
4,inductive,1687,0.0059
5,opaque,173,0.0006
6,quotient,3,0.0000
7,axiom,3,0.0000


## 1. Calculate PageRank, Personalized PageRank (PPR)

### 1-1. Verify PageRank
Check the correlation coefficient between the PageRank calculated in the papers in the dataset and the PageRank we calculated ourselves.

In [11]:
calculated_pr = nx.pagerank(G, alpha=0.85)

df_nodes['nx_pagerank'] = df_nodes['name'].map(calculated_pr)

# Calculate correlation between PageRanks
corr, _ = pearsonr(df_nodes['pagerank'].fillna(0), df_nodes['nx_pagerank'].fillna(0))
print(f"Correlation coefficient between existing and calculated PageRank: {corr:.4f}")
if corr > 0.95:
    print("Valid: Graph was successfully constructed!")
else:
    print("Invalid: Something is wrong.")

# Check whether sum of PageRank is 1 or not.
print(f"Original PageRank Sum: {df_nodes['pagerank'].sum()}")
print(f"Calculated PageRank Sum: {df_nodes['nx_pagerank'].sum()}")

# Drop calculated PageRank (For memory usage)
df_nodes.drop(columns=['nx_pagerank'], inplace=True)

Correlation coefficient between existing and calculated PageRank: 1.0000
Valid: Graph was successfully constructed!
Original PageRank Sum: 1.000000000000014
Calculated PageRank Sum: 1.000000000000038


### 1-2. PPR calculation for a specific field
By identifying which domain's PPR results a specific concept ranks highest, you can inversely map it to which field it is most closely associated with.

### 1-3. PageRank for subgraph of each domain
Find logical hierarchical structure within the domain.

In [12]:
# Find top-domain (Ex: A.B.C.D -> A)
df_nodes['top_domain'] = df_nodes['name'].apply(
    lambda x: str(x).split('.')[0] if pd.notnull(x) else 'Unknown'
)

# Check major domain distribution
# print(df_nodes['top_domain'].value_counts())
print(df_nodes['top_domain'].value_counts().head(10))

top_domain
CategoryTheory       38970
MeasureTheory         8194
List                  6168
Set                   5724
Nat                   5242
Finset                4607
AlgebraicGeometry     4318
Polynomial            3940
Int                   3277
SimpleGraph           2966
Name: count, dtype: int64


In [76]:
# Category classification
CATEGORIES = {
    "Topology_Core": [
        "Topology",
        "TopologicalSpace",
        "Filter",
        "UniformSpace",
        "PseudoMetricSpace",
        "MetricSpace",
        "PseudoEMetricSpace",
        "EMetricSpace",
        "Metric",
        "EMetric",
        "Bornology",
        "ContinuousMap",
        "Homeomorph",
    ],

    "Abstract_Algebra": [
        "Group",
        "Subgroup",
        "AddSubgroup",
        "Monoid",
        "Submonoid",
        "AddMonoid",
        "Subsemigroup",
        "Semigroup",
        "Ring",
        "Semiring",
        "CommRing",
        "Field",
        "Subfield",
        "Ideal",
        "Subring",
        "RingHom",
        "MonoidHom",
        "MulAction",
        "Algebra",
        "Subalgebra",
        "AlgHom",
        "AlgEquiv",
        "LieAlgebra",
        "LieSubalgebra",
        "LieModule",
        "LieIdeal",
    ],

    "Linear_Algebra_Modules": [
        "Module",
        "Submodule",
        "LinearMap",
        "LinearEquiv",
        "LinearIndependent",
        "Matrix",
        "TensorProduct",
        "Basis",
        "Finsupp",
        "DFinsupp",
        "ContinuousLinearMap",
    ],

    "Analysis_Normed_Calculus": [
        "Real",
        "Complex",
        "NNReal",
        "ENNReal",
        "EReal",
        "NormedSpace",
        "NormedField",
        "NormedRing",
        "SeminormedRing",
        "HasDerivAt",
        "HasDerivWithinAt",
        "HasFDerivAt",
        "HasFDerivWithinAt",
        "HasStrictDerivAt",
        "HasStrictFDerivAt",
    ],

    "Measure_Probability": [
        "MeasureTheory",
        "ProbabilityTheory",
        "MeasurableSpace",
        "Measurable",
        "MeasurableSet",
        "MeasurableEquiv",
        "AEMeasurable",
        "Measure",
    ],

    "Order_Lattice": [
        "Preorder",
        "PartialOrder",
        "LinearOrder",
        "OrderIso",
        "OrderEmbedding",
        "OrderHom",
        "Lattice",
        "CompleteLattice",
        "BooleanAlgebra",
        "HeytingAlgebra",
        "OrderDual",
        "WithTop",
        "WithBot",
    ],

    "Finite_Combinatorial_Graph": [
        "Fin",
        "Finset",
        "Fintype",
        "Multiset",
        "SimpleGraph",
        "Graph",
        "Matroid",
    ],

    "Number_Polynomial": [
        "Nat",
        "Int",
        "Rat",
        "ZMod",
        "Padic",
        "NumberField",
        "IsDedekindDomain",
        "ArithmeticFunction",
        "Polynomial",
        "MvPolynomial",
        "PowerSeries",
        "LaurentPolynomial",
    ],

    "CategoryTheory_Stress": [
        "CategoryTheory",
        "SSet",
        "TopCat",
        "ModuleCat",
        "Functor",
    ],
}

print(f"Total {len(CATEGORIES)} mathematics fields.")

Total 9 mathematics fields.


In [14]:
# Calculate PPR, subgraph PR for each categories
# start_time = time.time()

for cname, dlist in CATEGORIES.items():
    # Extract valid node list
    category_nodes = df_nodes[df_nodes['top_domain'].isin(dlist)]['name'].tolist()
    valid_nodes = [node for node in category_nodes if node in G]
    # print(valid_nodes[:25])

    if not valid_nodes:
        print(f"[{cname}] No valid nodes.")
        continue

    # ----------------------------------------
    # 1-2. PPR calculation for specific fields
    # ----------------------------------------
    print(f"[{cname}] Calculating PPR: {len(valid_nodes)} target nodes.")
    # Personalization weight (1 if target domain, else 0)
    personalization_dict = {node: 1.0 for node in valid_nodes}

    # PPR (Due to time limit, set tolerance = 1e-4)
    ppr_result = nx.pagerank(G, alpha=0.85, personalization=personalization_dict, weight=None, tol=1e-4)

    # Save PPR for each domain to new column (GNN feature)
    # Ex: 'ppr_Topology', 'ppr_Algebra'
    df_nodes[f'ppr_{cname}'] = df_nodes['name'].map(ppr_result)

    # ----------------------------------------
    # 1-3. PageRank for subgraph of each domain
    # ----------------------------------------
    print(f"Calculate PageRank for [{cname}] subgraph of {len(valid_nodes)} nodes...")

    # Create subgraph for each domain
    subgraph = G.subgraph(valid_nodes)

    # Calculate PageRank for each subgraph
    internal_pr = nx.pagerank(subgraph, alpha=0.85, weight=None, tol=1e-4)

    # save results in df_nodes['subgraph_pr'] -----------------
    # Map PR score for each name (NaN for unmapped nodes)
    mapped_scores = df_nodes['name'].map(internal_pr)

    # Filter NaN values
    valid_mask = mapped_scores.notna()

    # Update 'subgraph_pr' column
    # Note: subgraph_pr is stored in one shared column and used only for qualitative inspection.
    df_nodes.loc[valid_mask, 'subgraph_pr'] = mapped_scores[valid_mask]

# print(f"Complete PPR calculation! (Running time: {time.time() - start_time:.2f}seconds)\n")

[Topology_Core] Calculating PPR: 7468 target nodes.
Calculate PageRank for [Topology_Core] subgraph of 7468 nodes...
[Abstract_Algebra] Calculating PPR: 10616 target nodes.
Calculate PageRank for [Abstract_Algebra] subgraph of 10616 nodes...
[Linear_Algebra_Modules] Calculating PPR: 12623 target nodes.
Calculate PageRank for [Linear_Algebra_Modules] subgraph of 12623 nodes...
[Analysis_Normed_Calculus] Calculating PPR: 7606 target nodes.
Calculate PageRank for [Analysis_Normed_Calculus] subgraph of 7606 nodes...
[Measure_Probability] Calculating PPR: 11671 target nodes.
Calculate PageRank for [Measure_Probability] subgraph of 11671 nodes...
[Order_Lattice] Calculating PPR: 1608 target nodes.
Calculate PageRank for [Order_Lattice] subgraph of 1608 nodes...
[Finite_Combinatorial_Graph] Calculating PPR: 12568 target nodes.
Calculate PageRank for [Finite_Combinatorial_Graph] subgraph of 12568 nodes...
[Number_Polynomial] Calculating PPR: 17301 target nodes.
Calculate PageRank for [Number_P

In [81]:
# Inspect top subgraph PageRank nodes in selected category

# category = 'Topology_Core'
# category = 'Abstract_Algebra'
# category = 'Linear_Algebra_Modules'
# category = 'Analysis_Normed_Calculus'
# category = 'Measure_Probability'
# category = 'Order_Lattice'
# category = 'Finite_Combinatorial_Graph'
# category = 'Number_Polynomial'
category = 'CategoryTheory_Stress'

TOP_K = 20

dlist = CATEGORIES[category]

category_nodes = set(
    df_nodes[df_nodes['top_domain'].isin(dlist)]['name'].astype(str)
)

display(
    df_nodes[df_nodes['name'].astype(str).isin(category_nodes)]
    .sort_values('subgraph_pr', ascending=False)
    .head(TOP_K)
    [['name', 'kind', 'module', 'pagerank', 'subgraph_pr']]
)

,name,kind,module,pagerank,subgraph_pr
4329,CategoryTheory.Functor.obj,abbrev,CategoryTheory.Functor,0.001946,0.021414
4304,CategoryTheory.CategoryStruct.comp,abbrev,CategoryTheory.CategoryStruct,0.001564,0.019935
4330,CategoryTheory.Functor.map,abbrev,CategoryTheory.Functor,0.001577,0.019055
4336,CategoryTheory.NatTrans.app,abbrev,CategoryTheory.NatTrans,0.001412,0.018963
21890,CategoryTheory.Iso.hom,abbrev,CategoryTheory.Iso,0.001405,0.015479
21891,CategoryTheory.Iso.inv,abbrev,CategoryTheory.Iso,0.001289,0.013887
4337,CategoryTheory.Category.assoc,theorem,CategoryTheory.Category,0.000663,0.011132
4305,CategoryTheory.CategoryStruct.id,abbrev,CategoryTheory.CategoryStruct,0.000902,0.008828
21861,CategoryTheory.Functor.comp,definition,CategoryTheory.Functor,0.000510,0.004381
4328,CategoryTheory.Functor.mk,constructor,CategoryTheory.Functor,0.001148,0.003879


In [84]:
# PPR Top-k overlap matrix

TOP_K = 200

# Select category-wise PPR columns only.
# These columns were created in the previous PPR calculation cell.
# Example: ppr_Topology_Core, ppr_Abstract_Algebra, ...
ppr_cols = [
    c for c in df_nodes.columns
    if c.startswith('ppr_')
]

# Convert column names into clean category names.
# Example: ppr_Topology_Core -> Topology_Core
category_names = [c.replace('ppr_', '') for c in ppr_cols]

# Store the Top-K node set for each category.
topk_sets = {}

for col, cname in zip(ppr_cols, category_names):
    topk_sets[cname] = set(
        df_nodes
        .sort_values(col, ascending=False)
        .head(TOP_K)['name']
        .astype(str)
    )

# Create an empty category x category matrix.
overlap_matrix = pd.DataFrame(index=category_names, columns=category_names)

# Fill each cell with the Top-K overlap ratio.
for c1 in category_names:
    for c2 in category_names:
        overlap_matrix.loc[c1, c2] = len(topk_sets[c1] & topk_sets[c2]) / TOP_K

# Convert values to float for formatting and coloring.
overlap_matrix = overlap_matrix.astype(float)

# Display the matrix as a styled table.
display(
    overlap_matrix.style
    .format('{:.2f}')
    .background_gradient(axis=None)
)

,Topology_Core,Abstract_Algebra,Linear_Algebra_Modules,Analysis_Normed_Calculus,Measure_Probability,Order_Lattice,Finite_Combinatorial_Graph,Number_Polynomial,CategoryTheory_Stress
Topology_Core,1.00,0.99,0.99,0.99,0.99,0.98,0.99,0.99,0.99
Abstract_Algebra,0.99,1.00,0.99,0.99,0.99,0.98,0.99,0.99,0.99
Linear_Algebra_Modules,0.99,0.99,1.00,1.00,1.00,0.98,1.00,0.99,0.99
Analysis_Normed_Calculus,0.99,0.99,1.00,1.00,1.00,0.98,1.00,0.99,0.99
Measure_Probability,0.99,0.99,1.00,1.00,1.00,0.98,1.00,0.99,0.99
Order_Lattice,0.98,0.98,0.98,0.98,0.98,1.00,0.98,0.98,0.98
Finite_Combinatorial_Graph,0.99,0.99,1.00,1.00,1.00,0.98,1.00,0.99,0.99
Number_Polynomial,0.99,0.99,0.99,0.99,0.99,0.98,0.99,1.00,0.99
CategoryTheory_Stress,0.99,0.99,0.99,0.99,0.99,0.98,0.99,0.99,1.00


In [87]:
# Save PPR, subgraph PR features to csv
# df_nodes.to_csv("nodes_with_ppr.csv", index=False)

### 1-4. Relative PageRank for each domain
Reduce the effect of globally foundational declarations and highlight nodes that are relatively more domain-specific.

In [85]:
# Calculate relative PPR and show Relative PPR Top-k overlap matrix

TOP_K = 200
eps = 1e-12

# Select only category-wise PPR columns
ppr_cols = [
    c for c in df_nodes.columns
    if c.startswith('ppr_')
]

# Calculate relative PPR
for ppr_col in ppr_cols:
    cname = ppr_col.replace('ppr_', '')
    df_nodes[f'rel_ppr_{cname}'] = (
        np.log(df_nodes[ppr_col].fillna(0.0) + eps)
        - np.log(df_nodes['pagerank'].fillna(0.0) + eps)
    )

# Relative PPR Top-k overlap matrix
rel_cols = [c for c in df_nodes.columns if c.startswith('rel_ppr_')]
category_names = [c.replace('rel_ppr_', '') for c in rel_cols]

topk_sets = {}

for col, cname in zip(rel_cols, category_names):
    topk_sets[cname] = set(
        df_nodes.sort_values(col, ascending=False).head(TOP_K)['name'].astype(str)
    )

rel_overlap_matrix = pd.DataFrame(index=category_names, columns=category_names)

for c1 in category_names:
    for c2 in category_names:
        rel_overlap_matrix.loc[c1, c2] = len(topk_sets[c1] & topk_sets[c2]) / TOP_K

rel_overlap_matrix = rel_overlap_matrix.astype(float)

display(
    rel_overlap_matrix.style
    .format('{:.2f}')
    .background_gradient(axis=None)
)

,Topology_Core,Abstract_Algebra,Linear_Algebra_Modules,Analysis_Normed_Calculus,Measure_Probability,Order_Lattice,Finite_Combinatorial_Graph,Number_Polynomial,CategoryTheory_Stress
Topology_Core,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Abstract_Algebra,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Linear_Algebra_Modules,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00
Analysis_Normed_Calculus,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00
Measure_Probability,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00
Order_Lattice,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00
Finite_Combinatorial_Graph,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00
Number_Polynomial,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00
CategoryTheory_Stress,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00


In [89]:
# Relative PPR Top-k coverage in category premise space
TOP_K_LIST = [10, 50, 100, 500, 1000, 5000, 10000]

# Store premise nodes for each category
premise_space_by_category = {}

for cname, dlist in CATEGORIES.items():
    # Category declarations selected by top_domain
    category_node_set = set(
        df_nodes[df_nodes['top_domain'].isin(dlist)]['name'].astype(str)
    )

    # Edges whose source declaration belongs to this category
    category_edges = df_edges[
        df_edges['source'].astype(str).isin(category_node_set)
    ]

    # Premise space = target declarations referred to by category declarations
    premise_space_by_category[cname] = set(category_edges['target'].astype(str))

rows = []

for cname in CATEGORIES.keys():
    rel_col = f'rel_ppr_{cname}'

    # Skip if relative PPR for this category was not computed
    if rel_col not in df_nodes.columns:
        continue

    for k in TOP_K_LIST:
        # Top-k declarations by relative PPR score
        top_nodes = set(
            df_nodes.sort_values(rel_col, ascending=False).head(k)['name'].astype(str)
        )

        # Count how many top-k nodes are actually in the premise space
        overlap = top_nodes & premise_space_by_category[cname]
        coverage = len(overlap) / k

        rows.append({
            'category': cname,
            'top_k': k,
            'coverage': coverage,
            'overlap_count': len(overlap),
            'premise_space_size': len(premise_space_by_category[cname])
        })

# Convert results into category x top-k matrix
coverage_df = pd.DataFrame(rows)

coverage_matrix = coverage_df.pivot(
    index='category',
    columns='top_k',
    values='coverage'
)

# Display coverage ratio table
display(
    coverage_matrix.style
    .format('{:.3f}')
    .background_gradient(axis=None)
)

top_k,10,50,100,500,1000,5000,10000
category,,,,,,,
Abstract_Algebra,0.000,0.000,0.000,0.000,0.000,0.120,0.497
Analysis_Normed_Calculus,0.000,0.000,0.000,0.000,0.000,0.377,0.463
CategoryTheory_Stress,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Finite_Combinatorial_Graph,0.000,0.000,0.000,0.000,0.000,0.000,0.417
Linear_Algebra_Modules,0.000,0.000,0.000,0.000,0.000,0.031,0.434
Measure_Probability,0.000,0.000,0.000,0.000,0.000,0.237,0.601
Number_Polynomial,0.000,0.000,0.000,0.000,0.000,0.000,0.318
Order_Lattice,0.000,0.000,0.000,0.000,0.173,0.171,0.104
Topology_Core,0.000,0.000,0.000,0.000,0.000,0.293,0.410


## 2. Design GNN
### 2-1. Feature processing
Extract features and create node feature vector.
- Numeric features
- Categorical feature
- Text embedding

In [17]:
# Numerical feature normalization (Scale isn't same)
# Extremely skewed distribution -> log scale
skewed_cols = ['in_degree', 'out_degree']
df_nodes[skewed_cols] = np.log1p(df_nodes[skewed_cols].fillna(0))

# dag_layer: distance between each node and root
# If cannot be represented as dag, set to -1. (not a layer preceding 0, just an exception)
# Treat -1 as NaN, replace it with the mean/median of the other results
df_nodes['dag_layer'] = df_nodes['dag_layer'].replace(-1, np.nan)
mean_layer = df_nodes['dag_layer'].mean()
df_nodes['dag_layer'] = df_nodes['dag_layer'].fillna(mean_layer)

num_cols = skewed_cols + ['pagerank', 'betweenness', 'dag_layer']

# Change distribution to N(0, 1)
scaler = StandardScaler()
X_numeric = scaler.fit_transform(df_nodes[num_cols].fillna(0)).astype(np.float32)

# Categorical feature -> One-Hot Encoding
X_categorical = pd.get_dummies(df_nodes['kind'], dummy_na=False).values.astype(np.float32)

In [18]:
# Text embedding
# start_time = time.time()

# Environment settings
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"System: {device.upper()}")

# ==========================================
# Method 1 (Using sentence): Convert node name, kind, module as sentence
# Step 1: Generating natural and meaningful sentences
# def create_semantic_sentence(row):
#     name = str(row['name']).replace('.', ' ')
#     kind = str(row['kind']) if pd.notna(row['kind']) else "declaration"
#     namespace = str(row['namespace_depth2']).replace('.', ' ')

#     # Convert dots (.) to spaces for a natural look
#     # Basic sentence structure
#     sentence = f"'{name}' is a {kind} in namespace {namespace}"

#     # Module processing (add only value exists)
#     if 'module' in row and pd.notna(row['module']):
#         module = str(row['module']).replace('.', ' ')
#         sentence += f" from module {module}."
#     else:
#         sentence += "."

#     return sentence

# # Convert node data into natural language sentences
# # Apply the function row by row (axis=1)
# sentences_to_embed = df_nodes.apply(create_semantic_sentence, axis=1).tolist()

# # Example sentence
# # print("\n[3 example sentences]")
# # for s in sentences_to_embed[:3]:
# #     print("-", s)

# # Step 2: Sentence-BERT embedding (384-dim)
# model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# # Extract using batch size 256
# raw_embeddings = model.encode(sentences_to_embed, show_progress_bar=True, batch_size=256)
# print(f"Original embedding dim: {raw_embeddings.shape}")

# # Step 3: PCA dimensionality reduction (384 -> 64)
# pca = PCA(n_components=64, random_state=42)
# reduced_embeddings = pca.fit_transform(raw_embeddings)

# # Check Explained Variance Ratio
# explained_variance = np.sum(pca.explained_variance_ratio_) * 100
# print(f"-> Reduced dim: {reduced_embeddings.shape}")
# print(f"-> Original data information (variation) retention rate: Approx. {explained_variance:.2f}%\n")

# X_semantic = reduced_embeddings.astype(np.float32)
# ==========================================
# Method 2 (Equation): Convert equation ('type' column) of each node to semantic embedding
# CodeBERT: Extract embedding
model_code = SentenceTransformer('microsoft/codebert-base', device=device)

equations = df_nodes['type'].tolist()
# (N, 768) dim
rich_embeddings = model_code.encode(equations, batch_size=128, show_progress_bar=True, convert_to_numpy=True)

# PCA dimension reduction (768 -> 64)
pca_64 = PCA(n_components=64, random_state=42)
semantic_embeddings_64d = pca_64.fit_transform(rich_embeddings)

# Check Explained Variance Ratio
print(f"PCA EVR: {sum(pca_64.explained_variance_ratio_) * 100:.4f}")
print(f"Semantic embedding shape: {semantic_embeddings_64d.shape}")

X_semantic = semantic_embeddings_64d.astype(np.float32)
# ==========================================

System: CUDA


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Batches:   0%|          | 0/2222 [00:00<?, ?it/s]

PCA EVR: 92.3432
Semantic embedding shape: (284304, 64)


In [19]:
# Save results
# df_nodes.to_parquet('df_nodes.parquet')
# df_edges.to_parquet('df_edges.parquet')
# np.save("X_numeric.npy", X_numeric)
# np.save("X_categorical", X_categorical)
# np.save("X_semantic.npy", X_semantic)

# Reload data
# data_path = '/content/drive/MyDrive/CS471/data'
# df_nodes = pd.read_parquet(os.path.join(data_path, "df_nodes.parquet"))
# df_edges = pd.read_parquet(os.path.join(data_path, "df_edges.parquet"))
# X_numeric = np.load(os.path.join(data_path, "X_numeric.npy"))
# X_categorical = np.load(os.path.join(data_path, "X_categorical.npy"))
# X_semantic = np.load(os.path.join(data_path, "X_semantic.npy"))

In [20]:
# Concatenate all features
# 5 numeric + 8 categorical + 64 semantic
X_final = np.hstack([X_numeric, X_categorical, X_semantic])
x_tensor = torch.tensor(X_final)

print(f"Node feature matrix complete! Dim: {x_tensor.shape}")

Node feature matrix complete! Dim: torch.Size([284304, 77])


### 2-2. Creating PyTorch Geometric (PyG) Graph Objects
Converting Pandas data into Data objects that PyG can understand to run GNNs

In [21]:
# Create PyG graph

# Map mode name -> int ID (0: N-1)
node_to_id = {name: i for i, name in enumerate(df_nodes['name'])}

# Filter invalids
valid_edges = df_edges[df_edges['source'].isin(node_to_id) & df_edges['target'].isin(node_to_id)]
src_ids = valid_edges['source'].map(node_to_id).values
tgt_ids = valid_edges['target'].map(node_to_id).values

# Create edge_index (shape: [2, num_edges])
edge_index = torch.tensor([src_ids, tgt_ids], dtype=torch.long)

# PyG Data object
data = Data(x=x_tensor, edge_index=edge_index)
print(f"PyG graph generated: {data}")

PyG graph generated: Data(x=[284304, 77], edge_index=[2, 2178534])


/tmp/ipykernel_4334/2459245833.py:12: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  edge_index = torch.tensor([src_ids, tgt_ids], dtype=torch.long)


### 2-3. Data Splitting for Link Prediction
In link prediction, a portion of the edges (e.g., 10%) is masked, and the GNN is trained to detect these hidden edges through node embeddings. PyG's RandomLinkSplit automatically handles the sampling of positive and negative (false) edges.

In [22]:
# Split train/validation/test edges
transform = RandomLinkSplit(
    num_val=0.1,    # validation set: 10%
    num_test=0.1,   # test set: 10%
    is_undirected=False,    # Directed graph
    add_negative_train_samples=False    # Generate in minibatch during training
)

train_data, val_data, test_data = transform(data)
print(f"Training edges: {train_data.edge_label_index.size(1)}")
print(f"Validation edges: {val_data.edge_label_index.size(1)}")
print(f"Test edges: {test_data.edge_label_index.size(1)}")

Training edges: 1742828
Validation edges: 435706
Test edges: 435706


### 2-4. Design GNN Architecture (GraphSAGE + Dot Predictor)

Use GraphSAGE, which is memory-efficient and summarizes neighbor information well, as the encoder, and build a decoder that calculates connection probabilities through the dot product of two node vectors.

In [23]:
# Encoder: LayerNorm + Skip Connection
class Encoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

        # Normalize (Prevent loss explosion)
        self.norm1 = torch.nn.LayerNorm(hidden_channels)
        self.norm2 = torch.nn.LayerNorm(out_channels)

        # Linear layer for skip connection
        self.skip = torch.nn.Linear(in_channels, out_channels)

    def forward(self, x, edge_index):
        # Skip connection
        identity = self.skip(x)

        h = self.conv1(x, edge_index)
        h = self.norm1(h)
        h = F.relu(h)
        h = F.dropout(h, p=0.3, training=self.training)

        h = self.conv2(h, edge_index)
        h = self.norm2(h)

        # Combine original data +
        return h + identity


# Decoder: Directional MLP Decoder
class Decoder(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        # input: [src, tgt, src * tgt] = in_channels * 3
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(in_channels * 3, in_channels),
            torch.nn.LayerNorm(in_channels),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(in_channels, 1) # Logit
        )

    def forward(self, z, edge_label_index):
        src, tgt = edge_label_index

        z_src = z[src]
        z_tgt = z[tgt]

        # Concat + dot product (cosine similarity)
        # For directed edges
        z_combined = torch.cat([z_src, z_tgt, z_src * z_tgt], dim=-1)

        # 1-dim score
        return self.mlp(z_combined).squeeze(-1)


# Model
class LinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.encoder = Encoder(in_channels, hidden_channels, out_channels)
        self.decoder = Decoder(in_channels=out_channels)

    def forward(self, x, edge_index, edge_label_index):
        z = self.encoder(x, edge_index)
        return self.decoder(z, edge_label_index)

# Initialize model (input dim: column # of X_final)
# model = LinkPredictor(in_channels=data.num_features, hidden_channels=128, out_channels=64)

### 2-5. Mini-batch Training Loop
Reduce memory usage and prevent out of memory (OOM) problem, use PyG's LinkNeighborLoader to split the graph into subgraphs for training.

In [24]:
# Initialize model (input dim: column # of X_final)
model = LinkPredictor(in_channels=data.num_features, hidden_channels=128, out_channels=64)

# Set GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# --------------------------------------------
# 1. Set minibatch loader (Train & Validation)
# --------------------------------------------
train_loader = LinkNeighborLoader(
    train_data,
    num_neighbors=[10, 5],
    batch_size=1024,
    edge_label_index=train_data.edge_label_index,
    neg_sampling_ratio=1.0, # 1 negative edge per 1 positive edge
    shuffle=True
)
val_loader = LinkNeighborLoader(
    val_data,
    num_neighbors=[10, 5],
    batch_size=1024,
    edge_label_index=val_data.edge_label_index,
    edge_label=val_data.edge_label,
    # neg_sampling_ratio=1.0,
    shuffle=False
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = torch.nn.BCEWithLogitsLoss()

# --------------------------------
# 2. Parameter for Early stopping
# --------------------------------
MAX_EPOCHS = 60
PATIENCE = 12    # Terminate if loss increases 12 times in a row
best_val_loss = float('inf')
patience_counter = 0

print(f"[{device}] Start training model... (Max {MAX_EPOCHS} Epoch)")

# ----------------
# 3. Training Loop
# ----------------
for epoch in range(1, MAX_EPOCHS + 1):
    # epoch_start_time = time.time()
    # --- For Training ---
    model.train()
    total_train_loss = 0

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        # Model prediction
        out = model(batch.x, batch.edge_index, batch.edge_label_index)

        # Real label (Positive: 1, Negative: 0)
        target = batch.edge_label.float()

        # Calculate error & Backprop
        loss = criterion(out, target)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # --- For Validation ---
    model.eval() # Change mode
    total_val_loss = 0
    # Stop gradient calculation to save memory and increase speed
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.edge_label_index)
            target = batch.edge_label.float()

            loss = criterion(out, target)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    # Print results for each epoch
    # time_taken = time.time() - epoch_start_time
    print(f"Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # --- Early Stopping ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        # Safely save model weights at peak performance to a file
        # Backup of the smartest state
        torch.save(model.state_dict(), 'best_loss_model.pth')
        print("    Minimum Val Loss updated! Model saved.")
    else:
        patience_counter += 1
        print(f"    No Improvement (Patience: {patience_counter}/{PATIENCE})")

        if patience_counter >= PATIENCE:
            print(f"\n Early stop! Training finished at Epoch {epoch}.")
            break
    torch.save(model.state_dict(), 'final_model.pth')

print("Training complete")

[cuda] Start training model... (Max 60 Epoch)
Epoch 01 | Train Loss: 0.0604 | Val Loss: 0.1457
    Minimum Val Loss updated! Model saved.
Epoch 02 | Train Loss: 0.0343 | Val Loss: 0.1512
    No Improvement (Patience: 1/12)
Epoch 03 | Train Loss: 0.0313 | Val Loss: 0.1593
    No Improvement (Patience: 2/12)
Epoch 04 | Train Loss: 0.0300 | Val Loss: 0.1759
    No Improvement (Patience: 3/12)
Epoch 05 | Train Loss: 0.0294 | Val Loss: 0.1515
    No Improvement (Patience: 4/12)
Epoch 06 | Train Loss: 0.0292 | Val Loss: 0.1753
    No Improvement (Patience: 5/12)
Epoch 07 | Train Loss: 0.0288 | Val Loss: 0.1342
    Minimum Val Loss updated! Model saved.
Epoch 08 | Train Loss: 0.0284 | Val Loss: 0.1311
    Minimum Val Loss updated! Model saved.
Epoch 09 | Train Loss: 0.0285 | Val Loss: 0.1357
    No Improvement (Patience: 1/12)
Epoch 10 | Train Loss: 0.0280 | Val Loss: 0.1368
    No Improvement (Patience: 2/12)
Epoch 11 | Train Loss: 0.0282 | Val Loss: 0.1075
    Minimum Val Loss updated! Mode

## 3. Test model
### 3-1. Test Set Evaluation
Using test set, which the model has never seen even once, evaluate the model.
Here, ROC-AUC score, which evaluates the probability of a correct prediction, is used. (0.5 indicates a guess, 1.0 indicates a perfect prediction)

Also, calculate MRR, Hit@N for ranking evaluation for model.

In [59]:
# ROC-AUC (Area Under the Receiver Operating Characteristic Curve)

############################################################
# Choose model
model.load_state_dict(torch.load('best_loss_model.pth'))
# model.load_state_dict(torch.load('final_model.pth'))
#############################################################
model.eval()

# Minibatch for Test data (shuffle=False)
test_loader = LinkNeighborLoader(
    test_data,
    num_neighbors=[10, 5],
    batch_size=1024,
    edge_label_index=test_data.edge_label_index,
    edge_label=test_data.edge_label,
    # neg_sampling_ratio=1.0,
    shuffle=False
)

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)

        # Model prediction (returns Logit value)
        out = model(batch.x, batch.edge_index, batch.edge_label_index)

        # Convert to probability value (0~1) using Sigmoid function
        probs = torch.sigmoid(out).cpu().numpy()
        labels = batch.edge_label.cpu().numpy()

        all_preds.extend(probs)
        all_labels.extend(labels)

# Score
test_auc = roc_auc_score(all_labels, all_preds)
print(f"Test ROC-AUC Score: {test_auc:.4f}")

Test ROC-AUC Score: 0.9957


In [60]:
# MRR (Mean Reciprocal Rank), Hit@N

############################################################
# Choose model
model.load_state_dict(torch.load('best_loss_model.pth'))
# model.load_state_dict(torch.load('final_model.pth'))
#############################################################
model.eval()

with torch.no_grad():
    Z = model.encoder(
        train_data.x.to(device),
        train_data.edge_index.to(device)
    )

    pos_mask = test_data.edge_label == 1.0
    pos_test_edges = test_data.edge_label_index[:, pos_mask].to(device)

    src_nodes = pos_test_edges[0]
    tgt_nodes = pos_test_edges[1]

    num_test_total = src_nodes.size(0)

    # MLP decoder all-node ranking is expensive.
    # Limit to 2000 positives; increase if runtime is okay.
    # 여기 2000을 늘려보거나, min부분을 주석처리하고 전체에 대해서도 돌려보면 좋을듯.
    EVAL_LIMIT = min(10000, num_test_total)
    # EVAL_LIMIT = num_test_total
    # -------------------------------------
    src_nodes = src_nodes[:EVAL_LIMIT]
    tgt_nodes = tgt_nodes[:EVAL_LIMIT]

    num_test = src_nodes.size(0)
    num_nodes = Z.size(0)

    mrr_sum = 0.0
    hit_5_count = 0
    hit_10_count = 0
    hit_50_count = 0
    hit_100_count = 0

    batch_size = 8
    all_candidates = torch.arange(num_nodes, device=device)

    for i in range(0, num_test, batch_size):
        src_batch = src_nodes[i : i + batch_size]
        tgt_batch = tgt_nodes[i : i + batch_size]

        batch_ranks = []

        for src, tgt in zip(src_batch, tgt_batch):
            # candidate edges: src -> every candidate premise
            cand_src = src.repeat(num_nodes)
            cand_tgt = all_candidates

            cand_edge_index = torch.stack([cand_src, cand_tgt], dim=0)

            scores = model.decoder(Z, cand_edge_index)

            # exclude itself
            scores[src] = -float("inf")

            true_score = scores[tgt]
            rank = (scores > true_score).sum() + 1

            batch_ranks.append(rank)

        ranks = torch.stack(batch_ranks).float()

        mrr_sum += (1.0 / ranks).sum().item()
        hit_5_count += (ranks <= 5).sum().item()
        hit_10_count += (ranks <= 10).sum().item()
        hit_50_count += (ranks <= 50).sum().item()
        hit_100_count += (ranks <= 100).sum().item()

    mrr = mrr_sum / num_test
    hit_5 = hit_5_count / num_test
    hit_10 = hit_10_count / num_test
    hit_50 = hit_50_count / num_test
    hit_100 = hit_100_count / num_test

print(f"Evaluation portion: {num_test}/{num_test_total}")
print("-" * 50)
print(f"MRR      : {mrr:.4f}")
print(f"Hit@5    : {hit_5:.4f}")
print(f"Hit@10   : {hit_10:.4f}")
print(f"Hit@50   : {hit_50:.4f}")
print(f"Hit@100  : {hit_100:.4f}")
print("-" * 50)

Evaluation portion: 10000/217853
--------------------------------------------------
MRR      : 0.0839
Hit@5    : 0.1151
Hit@10   : 0.2110
Hit@50   : 0.4690
Hit@100  : 0.6017
--------------------------------------------------


### 3-2. Premise Recommendation & Evaluation
Use trained GNN model to recommend premises required to prove a specific theorem and evaluate its performance.
1. Global Embedding Extraction: Generates the embedding matrix for all nodes at once using the trained GNN encoder.
2. Candidate Pairing: Creates evaluation edge pairs by mapping the target theorem to all possible candidate premises.
3. Directional Scoring: Passes these candidate pairs through the trained decoder. This rigorously calculates the scores by distinguishing the direction.
4. Logical Filtering & Ranking: Applies strict masking to exclude invalid candidates—such as self-loops, already known training premises, and future nodes that violate the topological DAG hierarchy—before ranking and recommending the Top-K premises based on the highest scores.

In [61]:
############################################################
# Choose model
model.load_state_dict(torch.load('best_loss_model.pth'))
# model.load_state_dict(torch.load('final_model.pth'))
#############################################################

id_to_node = {i: name for name, i in node_to_id.items()}

# Prepare filtering objects for premise suggestion
dag_layer_tensor = torch.tensor(
    df_nodes["dag_layer"].fillna(-1).values,
    dtype=torch.long
).to(device)

known_train_premises = {}

train_pos_mask = train_data.edge_label == 1.0
train_pos_edges = train_data.edge_label_index[:, train_pos_mask]

for src, tgt in train_pos_edges.t().tolist():
    if src not in known_train_premises:
        known_train_premises[src] = set()
    known_train_premises[src].add(tgt)

print("known_train_premises built:", len(known_train_premises))

known_train_premises built: 255543


In [62]:
# Helpers for premise suggestion test
def suggest_premises(
    target_name,
    top_k=10,
    remove_known_premises=True,
    use_dag_filter=False,
    allow_same_layer=True
):
    if target_name not in node_to_id:
        return f"Error: Cannot find '{target_name}' in graph."

    target_idx = node_to_id[target_name]
    model.eval()

    print(f"Finding premises for [{target_name}]...")

    with torch.no_grad():
        Z = model.encoder(
            train_data.x.to(device),
            train_data.edge_index.to(device)
        )

        num_nodes = Z.size(0)
        all_candidates = torch.arange(num_nodes, device=device)

        # candidate edge: target declaration -> candidate premise
        src = torch.full(
            (num_nodes,),
            target_idx,
            dtype=torch.long,
            device=device
        )
        tgt = all_candidates

        candidate_edge_index = torch.stack([src, tgt], dim=0)

        scores = model.decoder(Z, candidate_edge_index)

        # exclude itself
        scores[target_idx] = -float("inf")

        # remove known train premises
        if remove_known_premises and "known_train_premises" in globals():
            known_premises = known_train_premises.get(target_idx, set())
            for premise_idx in known_premises:
                scores[premise_idx] = -float("inf")

        # dag_layer filter
        if use_dag_filter and "dag_layer_tensor" in globals():
            source_layer = dag_layer_tensor[target_idx].to(scores.device)
            layers = dag_layer_tensor.to(scores.device)

            if allow_same_layer:
                valid_dag_mask = layers >= source_layer
            else:
                valid_dag_mask = layers > source_layer

            scores[~valid_dag_mask] = -float("inf")

        top_scores, top_indices = torch.topk(scores, top_k)

    print(f"\n Suggested Top {top_k} Premises:")
    print("-" * 80)

    for rank, (score, idx) in enumerate(zip(top_scores, top_indices), 1):
        idx_int = idx.item()
        node_name = id_to_node[idx_int]
        raw_score = score.item()

        print(f"{rank:2d} | {node_name:45s} | score: {raw_score:.4f}")

    return top_indices


# Return the set of true premises used by target_name
def get_true_premises(target_name, edge_df=df_edges):
    true_premises = (
        edge_df[edge_df["source"].astype(str) == str(target_name)]["target"]
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    return set(true_premises)


# Run suggest_premises and compare recommended premises with true premises.
# DAG filter assumes that valid premises should be located at the same or deeper foundational layer.
def evaluate_premise_recommendation(
    target_name,
    top_k=50,
    remove_known_premises=False,
    use_dag_filter=True,
    allow_same_layer=True,
    show_true_premises=True
):
    if target_name not in node_to_id:
        print(f"Error: Cannot find '{target_name}' in graph.")
        return None

    # True premises from explicit/full edge dataframe
    true_premise_names = get_true_premises(target_name, edge_df=df_edges)

    print("=" * 80)
    print(f"Target declaration: {target_name}")
    print("=" * 80)
    print(f"Number of true premises: {len(true_premise_names)}")

    if show_true_premises:
        print("\nTrue premises:")
        for i, p in enumerate(sorted(true_premise_names), 1):
            print(f"{i:2d}. {p}")

    # Run GNN recommendation
    print("\n" + "=" * 80)
    print(f"GNN Top-{top_k} recommended premises")
    print("=" * 80)

    recommended_indices = suggest_premises(
        target_name,
        top_k=top_k,
        remove_known_premises=remove_known_premises,
        use_dag_filter=use_dag_filter,
        allow_same_layer=allow_same_layer
    )

    # Convert recommended indices to names
    recommended_names = []

    for idx in recommended_indices:
        idx_int = idx.item() if hasattr(idx, "item") else int(idx)
        recommended_names.append(id_to_node[idx_int])

    recommended_set = set(recommended_names)

    # Compare
    hits = recommended_set & true_premise_names

    hit_count = len(hits)
    hit_at_k = hit_count / top_k if top_k > 0 else 0.0
    recall_at_k = hit_count / len(true_premise_names) if len(true_premise_names) > 0 else 0.0

    print("\n" + "=" * 80)
    print("Evaluation")
    print("=" * 80)
    print(f"Hits in Top-{top_k}: {hit_count}")
    print(f"Hit ratio over Top-{top_k}: {hit_count}/{top_k} = {hit_at_k:.4f}")
    print(f"Recall over true premises: {hit_count}/{len(true_premise_names)} = {recall_at_k:.4f}")

    print("\nMatched premises:")
    if hit_count == 0:
        print("No matched premises.")
    else:
        for i, p in enumerate(sorted(hits), 1):
            print(f"{i:2d}. {p}")

    return {
        "target": target_name,
        "true_premises": true_premise_names,
        "recommended": recommended_names,
        "hits": hits,
        "hit_at_k": hit_at_k,
        "recall_at_k": recall_at_k,
    }

In [ ]:
# Run test
# Set target name
# Recommendation: "deriv_sin", "Matrix.det_mul", "Polynomial.degree_mul", "Nat.gcd_comm", "Continuous.comp"
target_node_name = "deriv_sin"

result = evaluate_premise_recommendation(
    target_node_name,
    top_k=50,
    remove_known_premises=False,
    use_dag_filter=True,
    allow_same_layer=True,
    show_true_premises=True
)

Target declaration: deriv_sin
Number of true premises: 3

True premises:
 1. DifferentiableAt.hasDerivAt
 2. HasDerivAt.deriv
 3. HasDerivAt.sin

GNN Top-50 recommended premises
Finding premises for [deriv_sin]...

 Suggested Top 50 Premises:
--------------------------------------------------------------------------------
 1 | Eq.refl                                       | score: 7.9416
 2 | Eq.symm                                       | score: 7.9159
 3 | OfNat.ofNat                                   | score: 7.7187
 4 | Iff.intro                                     | score: 7.6702
 5 | Eq.mpr                                        | score: 7.6046
 6 | DFunLike.coe                                  | score: 7.4518
 7 | Eq.trans                                      | score: 7.2903
 8 | of_eq_true                                    | score: 7.1931
 9 | HasDerivAt.sin                                | score: 7.1390
10 | HasDerivAt.cos                                | score: 7.0515
11 | p